# UC2 — Schema enforcement + multi-version from 2 assets

**Persona:** data ingestion pipeline owner enforcing input contracts.

**Goal:**
1. PATCH `items_schema` to declare `code` and `name` as mandatory.
2. Ingest a feature missing `code` → expect **HTTP 207 Multi-Status**
   with an `IngestionReport.rejections[*]` describing the missing field
   and a `policy_source` URL pointing back to the schema config.
3. Ingest features from **two different assets** with the same `code`
   value; the `ItemsWritePolicy.on_conflict=new_version` plus
   `enable_validity=true` create a new version row; the previous row's
   `valid_to` is bounded.


In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
RUN_ID = os.environ.get("RUN_ID", uuid.uuid4().hex[:8])
CATALOG_ID = os.environ.get("CATALOG_ID", f"cf_uc2_{RUN_ID}")
COLLECTION_ID = os.environ.get("COLLECTION_ID", f"col_{RUN_ID}")

IS_LOCAL = "localhost" in BASE_URL or "127.0.0.1" in BASE_URL

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120.0)

print(f"BASE_URL      : {BASE_URL}")
print(f"CATALOG_ID    : {CATALOG_ID}")
print(f"COLLECTION_ID : {COLLECTION_ID}")
print(f"AUTH          : {'token set' if TOKEN else 'anonymous'}")
if not TOKEN:
    print("\nWARNING: no Bearer token set — config writes will 401.")
    print("Set DYNASTORE_TOKEN before running write cells.")


In [ ]:
# Create catalog (idempotent: 409 = already exists)
catalog_payload = {
    "id": CATALOG_ID,
    "type": "Catalog",
    "title": f"Cycle F UC walkthrough {RUN_ID}",
    "description": "Ephemeral catalog for cycle_f_use_cases notebook.",
    "stac_version": "1.0.0",
}
r = client.post("/stac/catalogs", content=json.dumps(catalog_payload))
if r.status_code in (200, 201):
    print(f"Catalog '{CATALOG_ID}' created.")
elif r.status_code == 409:
    print(f"Catalog '{CATALOG_ID}' already exists.")
else:
    raise RuntimeError(f"Catalog create failed: {r.status_code}: {r.text}")

# Create collection (idempotent)
collection_payload = {
    "id": COLLECTION_ID,
    "type": "Collection",
    "stac_version": "1.0.0",
    "title": f"UC collection {RUN_ID}",
    "description": "Walkthrough collection — defaults inherited then PATCHed.",
    "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
    },
    "links": [],
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(collection_payload),
)
if r.status_code in (200, 201):
    print(f"Collection '{COLLECTION_ID}' created.")
elif r.status_code == 409:
    print(f"Collection '{COLLECTION_ID}' already exists.")
else:
    raise RuntimeError(f"Collection create failed: {r.status_code}: {r.text}")


In [ ]:
# Helper — show explicit-vs-effective config delta for a plugin
def show_config_delta(plugin_id: str, level: str = "collection") -> None:
    if level == "collection":
        base = f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
    elif level == "catalog":
        base = f"/configs/catalogs/{CATALOG_ID}"
    else:
        raise ValueError(level)
    rx = client.get(f"{base}/plugins/{plugin_id}")
    re_ = client.get(f"{base}/plugins/{plugin_id}/effective")
    print(f"\n=== {plugin_id} @ {level} ===")
    print("EXPLICIT:")
    if rx.status_code == 200:
        print(json.dumps(rx.json(), indent=2)[:600])
    else:
        print(f"  ({rx.status_code} — none stored, every field inherited)")
    if re_.status_code != 200:
        print(f"  effective unavailable ({re_.status_code})")
        return
    eff = re_.json()
    resolved = eff.get("resolved", eff.get("config", {}))
    sources = eff.get("sources", {})
    print("EFFECTIVE (resolved + per-field source):")
    for field in sorted(resolved.keys()):
        val = resolved[field]
        src = sources.get(field, "?")
        vs = json.dumps(val) if not isinstance(val, str) else val
        if len(str(vs)) > 70:
            vs = str(vs)[:67] + "..."
        marker = "*" if src == level else " "
        print(f"  {marker} {field:<30} {vs:<60} [{src}]")


In [ ]:
def put_config(plugin_id: str, body: dict, level: str = "collection") -> None:
    if level == "collection":
        url = f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/{plugin_id}"
    else:
        url = f"/configs/catalogs/{CATALOG_ID}/plugins/{plugin_id}"
    r = client.put(url, content=json.dumps(body), timeout=60.0)
    print(f"PUT {plugin_id}: {r.status_code}")
    if r.status_code not in (200, 201, 204):
        print(f"  body: {r.text[:300]}")
        if r.status_code == 401:
            raise RuntimeError("Unauthorized — set DYNASTORE_TOKEN.")
        raise RuntimeError(f"PUT failed: {r.status_code}")


## Step 1 — PATCH driver with attributes sidecar (external_id_field)

Sidecars before any rows; same immutability rule as UC1.


In [ ]:
items_pg_driver = {
    "engine_ref": "postgresql_engine_config",
    "sidecars": [
        {"sidecar_type": "geometries"},
        {
            "sidecar_type": "attributes",
            "enable_external_id": True,
            "external_id_field": "properties.code",
            "index_external_id": True,
            "enable_asset_id": True,
            "enable_validity": True,
        },
    ],
}
put_config("items_postgresql_driver_config", items_pg_driver)


## Step 2 — PATCH `items_schema` with mandatory fields

In [ ]:
schema_patch = {
    "fields": [
        {"name": "code", "type": "text", "required": True},
        {"name": "name", "type": "text", "required": True},
        {"name": "description", "type": "text"},
    ],
    "strict_unknown_fields": False,
}
put_config("items_schema", schema_patch)
show_config_delta("items_schema")


## Step 3 — PATCH `items_write_policy` for multi-version

In [ ]:
write_policy = {
    "on_conflict": "new_version",
    "identity_matchers": ["external_id"],
    "external_id_field": "properties.code",
    "track_asset_id": True,
    "enable_validity": True,
}
put_config("items_write_policy", write_policy)


## Step 4 — Ingest a feature missing `code` → 207 Multi-Status

Look at the `rejections` array — each entry includes `reason`,
`message`, `matcher` (which sidecar matcher fired), and a
`policy_source` URL pointing back to the schema config.


In [ ]:
bad = {
    "type": "Feature",
    "stac_version": "1.0.0",
    "id": f"bad-{RUN_ID}",
    "collection": COLLECTION_ID,
    "geometry": {"type": "Point", "coordinates": [12.5, 41.9]},
    "bbox": [12.5, 41.9, 12.5, 41.9],
    "properties": {
        "datetime": "2024-01-10T00:00:00Z",
        # NB: NO "code" — should be rejected
        "name": "Missing code field",
    },
    "assets": {},
    "links": [],
}
ingest_url = f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"
r = client.post(ingest_url, content=json.dumps(bad))
print(f"POST bad feature: {r.status_code}")
if r.status_code == 207:
    report = r.json()
    print("rejections:")
    for rej in report.get("rejections", []):
        print(f"  - reason={rej.get('reason')!r} matcher={rej.get('matcher')!r}")
        print(f"    message={rej.get('message')!r}")
        print(f"    policy_source={rej.get('policy_source')!r}")
elif r.status_code in (400, 422):
    print(f"  body: {r.text[:400]}")
    print("  (single-feature ingest may surface 400/422 instead of 207)")
else:
    print(f"  unexpected: {r.text[:300]}")


## Step 5 — Ingest 2 features with same `code` from different assets

First create two STAC assets (asset_id `pack-A` and `pack-B`).  Each
asset contributes a feature with `properties.code = "K42"`.  The first
ingest creates the row; the second triggers the `new_version` policy.


In [ ]:
# Helper to upload a tiny GeoJSON asset blob
def upload_asset(asset_id: str) -> str:
    url = f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/assets/{asset_id}"
    blob = {
        "type": "FeatureCollection",
        "features": [],
    }
    files = {"file": (f"{asset_id}.geojson", json.dumps(blob), "application/geo+json")}
    r = client.put(url, files=files, headers={k: v for k, v in client.headers.items()
                                              if k.lower() != "content-type"})
    print(f"  asset PUT {asset_id}: {r.status_code}")
    return asset_id

asset_a = upload_asset("pack-A")
asset_b = upload_asset("pack-B")

def ingest_with_asset(asset_id: str, idx: int) -> int:
    feat = {
        "type": "Feature",
        "stac_version": "1.0.0",
        "id": f"k42-via-{asset_id}-{RUN_ID}",
        "collection": COLLECTION_ID,
        "geometry": {"type": "Point", "coordinates": [12.5 + 0.01*idx, 41.9]},
        "bbox": [12.5 + 0.01*idx, 41.9, 12.5 + 0.01*idx, 41.9],
        "properties": {
            "datetime": "2024-01-10T00:00:00Z",
            "code": "K42",
            "name": f"From {asset_id}",
            "asset_id": asset_id,
        },
        "assets": {},
        "links": [],
    }
    r = client.post(ingest_url, content=json.dumps(feat))
    print(f"  ingest via {asset_id}: {r.status_code}")
    return r.status_code

s1 = ingest_with_asset(asset_a, 0)
s2 = ingest_with_asset(asset_b, 1)

# Read back items filtered by external_id="K42" — expect 2 rows when
# the new_version policy created a version row for the second asset.
search_url = f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"
r = client.get(search_url, params={"limit": 10, "external_id": "K42"})
rows = r.json().get("features", []) if r.status_code == 200 else []
print(f"\nK42 versions found: {len(rows)}")
for row in rows:
    props = row.get("properties", {})
    print(f"  geoid={row.get('id')} asset_id={props.get('asset_id')} valid_from={props.get('valid_from')} valid_to={props.get('valid_to')}")


## Teardown

In [ ]:
# Teardown — delete the ephemeral catalog. Comment out to keep state.
r = client.delete(
    f"/stac/catalogs/{CATALOG_ID}",
    params={"force": "true"},
    timeout=60.0,
)
print(f"teardown DELETE /stac/catalogs/{CATALOG_ID}: {r.status_code}")
client.close()
